# SVAMITVA Ensemble AI — DGX Training Pipeline (V3)

This notebook implements the production training pipeline for the SVAMITVA Feature Extraction Ensemble.

### Staged Training Strategy:
1. **Phase 1: Head Adaptation** — Encoders are frozen (`freeze_backbone_epochs=5`). Only specialized heads are trained.
2. **Phase 2: Global Fine-Tuning** — Encoders are unfrozen with a reduced learning rate (0.1x).

**Target Accuracy:** ≥95% IoU.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"
import sys
import torch
import logging
from pathlib import Path
import importlib
import shutil

# ── Environment Setup ──
MANUAL_REPO_ROOT = None

def find_repo_root(start_path, markers=["dgx_train.py", "aaronmodel"], max_depth=5):
    curr = Path(start_path).resolve()
    for _ in range(max_depth):
        for marker in markers:
            if (curr / marker).exists():
                if marker == "aaronmodel" and (curr / marker).is_dir():
                    return (curr / marker)
                return curr
        curr = curr.parent
    return Path(start_path).resolve()

if MANUAL_REPO_ROOT:
    REPO_ROOT = Path(MANUAL_REPO_ROOT).resolve()
else:
    try:
        REPO_ROOT = find_repo_root(os.getcwd())
    except:
        REPO_ROOT = Path(".").resolve()

os.chdir(str(REPO_ROOT))

# ── Remote Filesystem Recovery ──
old_path = REPO_ROOT / "training"
new_path = REPO_ROOT / "train_engine"
if old_path.exists() and not new_path.exists():
    os.rename(str(old_path), str(new_path))
    print("✅ Renamed training -> train_engine")

repo_path_str = str(REPO_ROOT)
if repo_path_str in sys.path:
    sys.path.remove(repo_path_str)
sys.path.insert(0, repo_path_str)
importlib.invalidate_caches()

logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")
logging.getLogger("train_engine").setLevel(logging.INFO)

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"GPUs: {torch.cuda.device_count()}")

In [ ]:
import subprocess, sys

needed_pkgs = ["tensordict", "hydra-core>=1.3.2", "iopath>=0.1.10"]
try:
    import tensordict
    print("✅ Core packages present.")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--user"] + needed_pkgs, check=True)

sam2_repo = REPO_ROOT / "sam2_source"
if str(sam2_repo) not in sys.path:
    sys.path.append(str(sam2_repo))

try:
    import sam2
    print("✅ SAM2 importable.")
except ImportError as e:
    print(f"❌ SAM2 failed: {e}")

sam2_ckpt = REPO_ROOT / "check" / "sam2.1_hiera_base_plus.pt"
if not sam2_ckpt.exists():
    sam2_ckpt.parent.mkdir(parents=True, exist_ok=True)
    import urllib.request
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt", sam2_ckpt)
    print("✅ SAM2 weights downloaded.")
else:
    print(f"✅ SAM2 weights found.")

In [ ]:
# ── Configuration ──
from train_engine.config import TrainingConfig

RUN_NAME = "universal_model_v4-5"

config = TrainingConfig(
    train_dirs=[
        Path("../DATA/MAP4"),
        Path("../DATA/MAP5"),
    ],
    batch_size=8,
    num_epochs=150,
    learning_rate=1e-4,
    mixed_precision=True,
    freeze_backbone_epochs=5,
    checkpoint_dir=Path(f"check/{RUN_NAME}"),
    num_workers=8,
    early_stopping=True,
    patience=30,
    sam2_checkpoint=REPO_ROOT / "check" / "sam2.1_hiera_base_plus.pt"
)

print(f"Training on: {[str(p) for p in config.train_dirs]}")
print(f"Batch Size: {config.batch_size}")
print(f"Checkpoint: {config.checkpoint_dir.absolute()}")

In [ ]:
# ── Data Loaders ──
from data.dataset import create_dataloaders

print("Initializing Data Loaders...")
train_loader, val_loader = create_dataloaders(
    train_dirs=config.train_dirs,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
    val_split=0.15,
    image_size=config.tile_size
)
print(f"Train Batches: {len(train_loader)} | Val Batches: {len(val_loader)}")

In [ ]:
# ── Model & Trainer ──
from models.model import EnsembleSvamitvaModel
from models.losses import MultiTaskLoss
from train_engine.trainer import Trainer

print("Building Ensemble Model...")
model = EnsembleSvamitvaModel(
    checkpoint_path=str(config.sam2_checkpoint),
    num_roof_classes=config.num_roof_classes,
    pretrained=True
)
loss_fn = MultiTaskLoss()

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    config=config
)
print("\nTraining Pipeline Ready.")

In [ ]:
# ── Start Training ──
try:
    trainer.fit()
except KeyboardInterrupt:
    print("\nTraining paused by user.")
except Exception as e:
    print(f"\nTraining error: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# ── Checkpoint Inspection ──
import torch
target_ckpt = config.checkpoint_dir / "best.pt"

if target_ckpt.exists():
    ckpt = torch.load(target_ckpt, map_location="cpu")
    print(f"Epoch: {ckpt.get('epoch', 'N/A')}")
    print(f"Best Score: {ckpt.get('best_score', 'N/A')}")
    print(f"Keys: {list(ckpt.keys())}")
else:
    print(f"best.pt not found at {target_ckpt}")

## 🎯 Custom YOLO Training (Point Features)

To train a custom YOLO model on your **Transformers, Tanks, and Wells**, run these two scripts in a terminal:

1. **Generate Dataset**: Converts shapefile points to YOLO bounding box format.
```bash
python scripts/prepare_yolo_dataset.py --map_dirs ../DATA/MAP4 ../DATA/MAP5 --output yolo_dataset
```

2. **Start Training**:
```bash
python scripts/train_yolo.py --data yolo_dataset/svamitva_points.yaml --epochs 100
```

The best weights will be saved to `check/yolo_svamitva_best.pt`.

## 🖼️ Inference on Standard Images (JPG/PNG)

You can now run the model on regular photos (not just GeoTIFF).

In [ ]:
# Example: Run inference on a JPG/PNG
from pathlib import Path
from inference.predict import load_ensemble_pipeline

IMG_PATH = Path("path/to/your/photo.jpg") # <-- Update this

if IMG_PATH.exists():
    predictor = load_ensemble_pipeline(
        weights_path="check/universal_model_v4-5/best.pt",
        device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )
    results = predictor.predict_image(IMG_PATH)
    print(f"Detected tasks: {list(results.keys())}")
else:
    print("Update IMG_PATH to test JPG inference.")

---
## 🚀 Multi-GPU DDP Training (Terminal Only)

For maximum speed (8 GPUs), run this in the terminal:

```bash
torchrun --nproc_per_node=8 dgx_train_ddp.py \\
    --train_dirs ../DATA/MAP4 ../DATA/MAP5 \\
    --batch_size 128 --num_workers 32
```